# MyLaw-Serve: CPU variants (PyTorch vs ONNX vs int8)

Use a **CPU runtime** (Runtime > Change runtime type > CPU). Run every variant in this one session, so the speed numbers
come from the same hardware and are comparable. Run the cells one at a time, top to bottom.

In [ ]:
REPO_URL = "https://github.com/husainn-7/mylaw-serve.git"

import os, subprocess, time, urllib.request, torch

print("GPU available:", torch.cuda.is_available(), "(fine, but these runs use the CPU only)")
if not os.path.isdir("/content/mylaw-serve"):
    subprocess.run(["git", "clone", REPO_URL, "/content/mylaw-serve"], check=True)
else:
    subprocess.run(["git", "-C", "/content/mylaw-serve", "pull"], check=True)
os.chdir("/content/mylaw-serve")
print(subprocess.run("lscpu | grep 'Model name'; nproc", shell=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements-onnx.txt pytest

In [ ]:
!python -m pytest -q

## Convert the two models to ONNX (also makes the int8 copies)

In [ ]:
!python -m mlserve.onnx_export

## Helpers

In [ ]:
def sh(cmd, env=None):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env or os.environ)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode:
        raise RuntimeError(f"command failed: {cmd}")


def run_engine(engine, runs=20, concurrency="1,2,4", duration=15):
    env = {**os.environ, "ENGINE": engine, "DEVICE": "cpu", "TOKENIZERS_PARALLELISM": "false"}
    sh(f"python bench/quality.py --engine {engine} --device cpu", env)
    sh(f"python bench/micro.py --engine {engine} --device cpu --runs {runs}", env)
    server = subprocess.Popen(["uvicorn", "mlserve.api:app", "--port", "8000", "--log-level", "warning"],
                              env=env, stdout=open(f"/tmp/server_{engine}.log", "w"), stderr=subprocess.STDOUT)
    try:
        for _ in range(300):
            try:
                urllib.request.urlopen("http://127.0.0.1:8000/healthz", timeout=2)
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError(f"server did not start, see /tmp/server_{engine}.log")
        sh(f"python bench/loadtest.py --concurrency {concurrency} --duration {duration}", env)
    finally:
        server.terminate()
        server.wait(timeout=15)

## Run each variant (one cell each; the PyTorch baseline first)

In [ ]:
run_engine("torch")

In [ ]:
run_engine("onnx")

In [ ]:
run_engine("onnx-int8-rerank")

In [ ]:
run_engine("onnx-int8")

## Side-by-side table, then download the results

In [ ]:
!python bench/report.py
!python bench/report.py --markdown

In [ ]:
!zip -r -q results.zip bench/results
from google.colab import files
files.download("results.zip")